In [1]:
# Phase 5 – Network Analysis | Cell 1: Setup + Load

from pathlib import Path
from datetime import datetime
import warnings
warnings.filterwarnings("ignore")

import pandas as pd
import numpy as np
import networkx as nx
import pickle

PROJECT = Path(r"D:\univercity\omid_project\economics_project")

OUT_DIR  = PROJECT / "outputs" / "phase5"
OUT_DATA = OUT_DIR / "data"
OUT_TAB  = OUT_DIR / "tables"
OUT_FIG  = OUT_DIR / "figures"
for d in [OUT_DATA, OUT_TAB, OUT_FIG]:
    d.mkdir(parents=True, exist_ok=True)

print("=" * 60)
print("Phase 5 – Network Analysis | Cell 1")
print("=" * 60)
print(f"Timestamp    : {datetime.now().strftime('%Y-%m-%d %H:%M:%S')}")
print(f"Project Root : {PROJECT}")
print(f"Outputs      : {OUT_DIR}")

with open(PROJECT / "outputs" / "phase4" / "data" / "temporal_trade_networks_clean.pkl", "rb") as f:
    networks = pickle.load(f)

print(f"\nLoaded networks: {len(networks)}")
print(f"Sample years  : {sorted(list(networks.keys()))[:5]}")
y0 = sorted(networks.keys())[0]
g0 = networks[y0]
print(f"Sample graph  : year={y0} | nodes={g0.number_of_nodes()} | edges={g0.number_of_edges()}")

print("\nCell 1 completed successfully.")

Phase 5 – Network Analysis | Cell 1
Timestamp    : 2026-09-27 08:06:00
Project Root : D:\univercity\omid_project\economics_project
Outputs      : D:\univercity\omid_project\economics_project\outputs\phase5

Loaded networks: 21
Sample years  : [2000, 2001, 2002, 2003, 2004]
Sample graph  : year=2000 | nodes=18 | edges=306

Cell 1 completed successfully.


In [2]:
# Phase 5 – Network Analysis | Cell 2: Centrality Measures

from pathlib import Path
import pandas as pd
import numpy as np
import networkx as nx
import pickle

PROJECT = Path(r"D:\univercity\omid_project\economics_project")
OUT_DATA = PROJECT / "outputs" / "phase5" / "data"
OUT_TAB  = PROJECT / "outputs" / "phase5" / "tables"

print("=" * 60)
print("Phase 5 – Network Analysis | Cell 2: Centrality")
print("=" * 60)

with open(PROJECT / "outputs" / "phase4" / "data" / "temporal_trade_networks_clean.pkl", "rb") as f:
    networks = pickle.load(f)

records = []
for y, g in sorted(networks.items(), key=lambda x: x[0]):
    # weight-aware strengths
    in_s = dict(g.in_degree(weight="weight"))
    out_s = dict(g.out_degree(weight="weight"))

    # for path-based metrics use inverse weight as distance
    g_dist = g.copy()
    for u, v, d in g_dist.edges(data=True):
        w = float(d.get("weight", 0.0))
        d["distance"] = 1.0 / w if w > 0 else 1e12

    try:
        betweenness = nx.betweenness_centrality(g_dist, weight="distance", normalized=True)
    except Exception:
        betweenness = {n: 0.0 for n in g.nodes()}

    try:
        closeness = nx.closeness_centrality(g_dist, distance="distance")
    except Exception:
        closeness = {n: 0.0 for n in g.nodes()}

    try:
        pagerank = nx.pagerank(g, weight="weight", max_iter=500, tol=1e-8)
    except Exception:
        pagerank = {n: 1.0 / g.number_of_nodes() for n in g.nodes()}

    try:
        # eigenvector on undirected projection for stability
        gu = g.to_undirected()
        eigenvector = nx.eigenvector_centrality_numpy(gu, weight="weight")
    except Exception:
        eigenvector = {n: 0.0 for n in g.nodes()}

    try:
        katz = nx.katz_centrality_numpy(g, weight="weight", alpha=1e-4)
    except Exception:
        katz = {n: 0.0 for n in g.nodes()}

    for n in g.nodes():
        records.append({
            "year": y,
            "iso2": n,
            "in_strength": float(in_s.get(n, 0.0)),
            "out_strength": float(out_s.get(n, 0.0)),
            "betweenness": float(betweenness.get(n, 0.0)),
            "closeness": float(closeness.get(n, 0.0)),
            "pagerank": float(pagerank.get(n, 0.0)),
            "eigenvector": float(eigenvector.get(n, 0.0)),
            "katz": float(katz.get(n, 0.0)),
        })

cent = pd.DataFrame(records)
print(f"Centrality records: {len(cent)}")
print(cent.head(10).to_string(index=False))

cent.to_pickle(OUT_DATA / "centrality_measures.pkl")
cent.to_csv(OUT_TAB / "centrality_measures.csv", index=False)
print(f"\nSaved → {OUT_DATA / 'centrality_measures.pkl'}")
print("\nCell 2 completed successfully.")

Phase 5 – Network Analysis | Cell 2: Centrality
Centrality records: 378
 year iso2  in_strength  out_strength  betweenness    closeness  pagerank  eigenvector      katz
 2000   AU 4.030550e+07  3.087409e+07     0.000000 6.685372e+06  0.020712     0.064860  0.165681
 2000   BE 1.469869e+08  1.185637e+08     0.000000 9.276756e+06  0.055814     0.171532 -0.393456
 2000   CA 1.939192e+08  2.462788e+08     0.000000 1.140357e+07  0.072914     0.435485 -0.174147
 2000   CH 7.270168e+07  6.350959e+07     0.000000 8.414299e+06  0.031421     0.094519 -0.073324
 2000   DE 2.984054e+08  3.633153e+08     0.643382 1.246021e+07  0.113045     0.325481  0.063802
 2000   DK 3.681385e+07  3.547126e+07     0.000000 5.634906e+06  0.020504     0.032506  0.258281
 2000   ES 1.172277e+08  8.719323e+07     0.058824 8.829396e+06  0.048547     0.114301  0.140162
 2000   FI 2.403255e+07  3.138068e+07     0.000000 3.766519e+06  0.016398     0.025122 -0.132393
 2000   FR 2.387396e+08  2.259199e+08     0.069853 1.10

In [3]:
# Phase 5 – Network Analysis | Cell 3: Structural + Resilience Metrics

from pathlib import Path
import pandas as pd
import numpy as np
import networkx as nx
import pickle

PROJECT = Path(r"D:\univercity\omid_project\economics_project")
OUT_DATA = PROJECT / "outputs" / "phase5" / "data"
OUT_TAB  = PROJECT / "outputs" / "phase5" / "tables"

print("=" * 60)
print("Phase 5 – Network Analysis | Cell 3: Structural/Resilience")
print("=" * 60)

with open(PROJECT / "outputs" / "phase4" / "data" / "temporal_trade_networks_clean.pkl", "rb") as f:
    networks = pickle.load(f)

rows = []
for y, g in sorted(networks.items(), key=lambda x: x[0]):
    gu = g.to_undirected()
    # distance for path metrics
    g_dist = g.copy()
    for u, v, d in g_dist.edges(data=True):
        w = float(d.get("weight", 0.0))
        d["distance"] = 1.0 / w if w > 0 else 1e12

    density = nx.density(g)
    try:
        clustering = nx.average_clustering(gu, weight="weight")
    except Exception:
        clustering = np.nan
    try:
        assort = nx.degree_assortativity_coefficient(g, weight="weight")
    except Exception:
        assort = np.nan
    try:
        geff = nx.global_efficiency(gu)
    except Exception:
        geff = np.nan
    try:
        leff = nx.local_efficiency(gu)
    except Exception:
        leff = np.nan
    try:
        # diameter on undirected unweighted completeness graph is 1 if complete
        if nx.is_connected(gu):
            diameter = nx.diameter(gu)
        else:
            diameter = np.nan
    except Exception:
        diameter = np.nan
    try:
        L = nx.normalized_laplacian_matrix(gu, weight="weight").astype(float)
        eigs = np.linalg.eigvalsh(L.toarray())
        eigs = np.sort(eigs)
        algebraic_connectivity = float(eigs[1]) if len(eigs) > 1 else np.nan
        spectral_gap = float(eigs[1] - eigs[0]) if len(eigs) > 1 else np.nan
    except Exception:
        algebraic_connectivity, spectral_gap = np.nan, np.nan

    rows.append({
        "year": y,
        "density": density,
        "diameter": diameter,
        "clustering": clustering,
        "assortativity": assort,
        "global_efficiency": geff,
        "local_efficiency": leff,
        "algebraic_connectivity": algebraic_connectivity,
        "spectral_gap": spectral_gap,
    })

struct = pd.DataFrame(rows)
print(struct.head(10).to_string(index=False))

struct.to_pickle(OUT_DATA / "structural_resilience_metrics.pkl")
struct.to_csv(OUT_TAB / "structural_resilience_metrics.csv", index=False)
print(f"\nSaved → {OUT_DATA / 'structural_resilience_metrics.pkl'}")
print("\nCell 3 completed successfully.")

Phase 5 – Network Analysis | Cell 3: Structural/Resilience
 year  density  diameter  clustering  assortativity  global_efficiency  local_efficiency  algebraic_connectivity  spectral_gap
 2000      1.0         1    0.034674      -0.054082                1.0               1.0                0.516108      0.516108
 2001      1.0         1    0.037070      -0.054189                1.0               1.0                0.516351      0.516351
 2002      1.0         1    0.038035      -0.052480                1.0               1.0                0.500535      0.500535
 2003      1.0         1    0.041547      -0.051370                1.0               1.0                0.503041      0.503041
 2004      1.0         1    0.043174      -0.050568                1.0               1.0                0.503584      0.503584
 2005      1.0         1    0.042235      -0.050733                1.0               1.0                0.498959      0.498959
 2006      1.0         1    0.043326      -0.051631 

In [4]:
# Phase 5 – Network Analysis | Cell 4: Feature Matrix

from pathlib import Path
import pandas as pd
import numpy as np

PROJECT = Path(r"D:\univercity\omid_project\economics_project")
OUT_DATA = PROJECT / "outputs" / "phase5" / "data"
OUT_TAB  = PROJECT / "outputs" / "phase5" / "tables"

print("=" * 60)
print("Phase 5 – Network Analysis | Cell 4: Feature Matrix")
print("=" * 60)

cent = pd.read_pickle(OUT_DATA / "centrality_measures.pkl")
struct = pd.read_pickle(OUT_DATA / "structural_resilience_metrics.pkl")
panel = pd.read_pickle(PROJECT / "outputs" / "phase2" / "data" / "panel_final_clean.pkl")
icio = pd.read_pickle(PROJECT / "outputs" / "phase4" / "data" / "icio_country_summary_clean.pkl")
cw = pd.read_pickle(PROJECT / "outputs" / "phase1" / "data" / "country_crosswalk.pkl")

# node-level centrality already long
fm = cent.copy()

# merge structural snapshot metrics
fm = fm.merge(struct, on="year", how="left")

# merge crosswalk for names/iso3
fm = fm.merge(cw[["iso2", "iso3", "country_jst"]], on="iso2", how="left")

# merge selected macro / credit / trade features
macro_cols = [c for c in [
    "year", "iso2", "rgdpmad", "gdp", "cpi", "stir", "ltrate", "debtgdp", "unemp", "iy",
    "credit_private_pct_gdp", "credit_gov_pct_gdp",
    "export_kusd", "import_kusd", "trade_openness_kusd", "ca"
] if c in panel.columns]
fm = fm.merge(panel[macro_cols], on=["year", "iso2"], how="left")

# merge ICIO summaries
fm = fm.merge(icio, on=["year", "iso3"], how="left")

fm = fm.sort_values(["year", "iso2"]).reset_index(drop=True)

print(f"Feature Matrix shape: {fm.shape}")
print(f"Columns ({fm.shape[1]}):")
print(list(fm.columns))
print("\n--- Sample ---")
print(fm.head(5).to_string(index=False))

fm.to_pickle(OUT_DATA / "feature_matrix.pkl")
fm.to_csv(OUT_TAB / "feature_matrix.csv", index=False)

print(f"\nSaved → {OUT_DATA / 'feature_matrix.pkl'}")
print("\nCell 4 completed successfully.")
print("PHASE 5 COMPLETED.")

Phase 5 – Network Analysis | Cell 4: Feature Matrix
Feature Matrix shape: (378, 36)
Columns (36):
['year', 'iso2', 'in_strength', 'out_strength', 'betweenness', 'closeness', 'pagerank', 'eigenvector', 'katz', 'density', 'diameter', 'clustering', 'assortativity', 'global_efficiency', 'local_efficiency', 'algebraic_connectivity', 'spectral_gap', 'iso3', 'country_jst', 'rgdpmad', 'gdp', 'cpi', 'stir', 'ltrate', 'debtgdp', 'unemp', 'iy', 'credit_private_pct_gdp', 'credit_gov_pct_gdp', 'export_kusd', 'import_kusd', 'trade_openness_kusd', 'ca', 'icio_output', 'icio_hfce', 'icio_ggfc']

--- Sample ---
 year iso2  in_strength  out_strength  betweenness    closeness  pagerank  eigenvector      katz  density  diameter  clustering  assortativity  global_efficiency  local_efficiency  algebraic_connectivity  spectral_gap iso3 country_jst      rgdpmad          gdp        cpi     stir   ltrate  debtgdp  unemp       iy  credit_private_pct_gdp  credit_gov_pct_gdp  export_kusd  import_kusd  trade_openne